## **How can I use this method: LIME in practice**

Here's the English translation for an open tutorial on LIME, formatted in Markdown:

If we delve into the details, LIME also performs (or can perform) interesting transformations that can enhance and specialize the explanation. Additionally, by considering the use of the method from different perspectives, it is possible to obtain several different explanation variants by using the hyperparameters of the library's methods.

Instead of using the LIME wrapper, in this practical exercise, we will understand everything and build two of our own surrogate models from scratch and recreate the approach from [this paper](https://arxiv.org/pdf/1910.13016). Going through this notebook will help you "feel" the algorithm and use it wisely.

Let's get started!

telegram: https://t.me/sabrina_sadiekh \
[LinkedIn](https://www.linkedin.com/in/sabrina-sadiekh-35181a286/) \
My course about explainable AI (rus): https://open-xai-platform.web.app \



[![temp-Image1mlt-QZ.avif](https://i.postimg.cc/LXny3rrP/temp-Image1mlt-QZ.avif)](https://postimg.cc/hzRbS344)




In [ ]:
!pip install lime fat-forensics[all] -q # install the required libraries

In [ ]:
import fatf
import lime
import pandas as pd
import numpy as np

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

fatf.setup_random_seed(42)

Let's prepare the data. For simplicity, we will work with a very well-known dataset in statistics — the Fisher's Iris dataset. It contains measurements taken from 150 iris specimens, with 50 specimens each from three species — Iris setosa, Iris virginica, and Iris versicolor.

For each specimen, four characteristics are provided (in centimeters):

- Sepal length
- Sepal width
- Petal length
- Petal width

Based on this dataset, we need to build a classification rule to determine the species of the plant from the given measurements. This is a multi-class classification problem since there are three classes — the three species of iris.


In [ ]:
#Data loading

iris_data_dict = load_iris()
iris_data = iris_data_dict['data']
iris_target = iris_data_dict['target']
iris_feature_names = iris_data_dict['feature_names']
iris_target_names = iris_data_dict['target_names']

X_train, X_test, y_train, y_test = train_test_split(iris_data, iris_target, random_state=42)

As the "black box" model, we will use a random forest.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import sklearn.metrics

blackbox_model = RandomForestClassifier(random_state=42, max_depth=2)
blackbox_model.fit(X_train, y_train)

predictions = blackbox_model.predict(X_test)
acc = sklearn.metrics.accuracy_score(y_test, predictions)

print(f'Model accuracy: {acc}')

In [ ]:
data_point = X_train[37] #Let's choose a random data point

data_point_probabilities = blackbox_model.predict_proba(data_point.reshape(1, -1))[0]
data_point_probabilities

In [ ]:
data_point

In [ ]:
data_point_prediction = data_point_probabilities.argmax(axis=0) #Looking at the class for the data point

data_point_class = iris_target_names[data_point_prediction]
data_point_class

Excellent! Let's consider the Iris versicolor. To literally take a look at it, we will visualize the dataset and the selected data point.



In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


_ = plt.figure()
_ = plt.scatter(
     X_train[y_train==0][:, 2],
     X_train[y_train==0][:, 3],
     label=iris_target_names[0])
_ = plt.scatter(
     X_train[y_train==1][:, 2],
     X_train[y_train==1][:, 3],
     label=iris_target_names[1])
_ = plt.scatter(
     X_train[y_train==2][:, 2],
     X_train[y_train==2][:, 3],
     label=iris_target_names[2])
_ = plt.scatter(
     data_point[2],
     data_point[3],
     label='Explained Data Point',
    s=100, c='k')

_ = plt.xlabel(iris_feature_names[2])
_ = plt.ylabel(iris_feature_names[3])
_ = plt.legend()

_ = plt.title('True and interpreted values')

## Build LIME model

Let's arm ourselves with some theoretical background about the algorithm. When training a surrogate model \( g(z) \) for the model \( f(x) \), we solve the following minimization problem:

$$\xi(x) = \arg\min_{g \in G} L(f, g, \pi_x) + \Omega(g),$$

where
- $G$ is the set of interpretable models, $g \in G $ is some interpretable model (linear regression, decision tree, etc.)
- $f(x)$ is the black box model (the one we want to explain)
- $\Omega(g)$ is the term that accounts for the complexity of the surrogate model.

It is important to note that the loss function $L$ includes an unconventional parameter $\pi_x$. This parameter adjusts the contribution of samples from the neighborhood, assigning them weights and acting as a correction term when computing the quadratic error.

$$L(f, g, \pi_x) = \pi_x(f(z) - g(x'))^2,$$

$\pi(x)$ for each $x$ is calculated according to the formula $\exp\left(\frac{-D(x', x)}{\sigma^2}\right)$ (where $D(., .)$ denotes distance, $\sigma$ is the width of the neighborhood kernel).

In the original implementation ([“Why Should I Trust You?” Explaining the Predictions of Any Classifier](https://arxiv.org/pdf/1602.04938)), the local surrogate model can be trained with or without *discretizing* continuous features, and for some data modalities (text, images), binarization is used. These concepts might be unfamiliar, but in practice, they allow us to view the constructed surrogate models from different perspectives and obtain different feature interpretations.


In [ ]:
import fatf.utils.data.discretisation as fatf_discretisation # import the helper functions
import fatf.utils.data.augmentation as fatf_augmentation

**Step 1.** The first step of the LIME algorithm is generating data similar to the original data.

In [ ]:
augmenter = fatf_augmentation.Mixup(X_train, ground_truth=y_train)

sampled_data = augmenter.sample(data_point, samples_number=100) # generated points based on the original ones
sampled_data_probabilities = blackbox_model.predict_proba(sampled_data) # predicted the probabilities for the generated points

Let's look at the new points.

In [ ]:
sampled_data_predictions = sampled_data_probabilities.argmax(axis=1)
sampled_data_0_indices = np.where(sampled_data_predictions == 0)[0]
sampled_data_1_indices = np.where(sampled_data_predictions == 1)[0]
sampled_data_2_indices = np.where(sampled_data_predictions == 2)[0]

_ = plt.figure()
_ = plt.scatter(
     X_train[y_train==0][:, 2],
     X_train[y_train==0][:, 3],
     label=iris_target_names[0])
_ = plt.scatter(
     X_train[y_train==1][:, 2],
     X_train[y_train==1][:, 3],
     label=iris_target_names[1])
_ = plt.scatter(
     X_train[y_train==2][:, 2],
     X_train[y_train==2][:, 3],
     label=iris_target_names[2])
_ = plt.scatter(
     data_point[2],
     data_point[3],
     label='Explained Data Point',
    s=100, c='k')


_ = plt.scatter(
     sampled_data[sampled_data_0_indices, 2],
     sampled_data[sampled_data_0_indices, 3],
     label='Augmented Data: {}'.format(iris_target_names[0]))
_ = plt.scatter(
     sampled_data[sampled_data_1_indices, 2],
     sampled_data[sampled_data_1_indices, 3],
     label='Augmented Data: {}'.format(iris_target_names[1]))
_ = plt.scatter(
     sampled_data[sampled_data_2_indices, 2],
     sampled_data[sampled_data_2_indices, 3],
     label='Augmented Data: {}'.format(iris_target_names[2]))

_ = plt.xlabel(iris_feature_names[2])
_ = plt.ylabel(iris_feature_names[3])
_ = plt.legend()

**Observations:**

Note that the synthetic points are **similar** to the original ones and are close to **all** classes. However, this is not the only approach to creating synthetic data for predictions. New data can also be generated **only in the vicinity** of the explained point. In the official Lime implementation, this is controlled by the hyperparameter `sample_around_instance`.

We will implement an approach where synthetic points are generated based on **all** the data.

```
**Remember:** the first thing you can change in LIME to get a different explanation is the vicinity of the point.
```



**Step 2. Discretization and Binarization of Data**

**Discretization** is the process of transforming a continuous function into a discrete one. In this case (and this is one of the possible implementations of the LIME algorithm out-of-the-box), discretization is done by splitting into *quartiles*.

**Quartiles** are values that divide an ordered dataset into four equal parts, each containing 25% of the data. The process of discretization (splitting) into quartiles is called quartile discretization. It occurs as follows:
1. Extract the boundaries of each quartile.
2. Encode each point in the dataset as a vector with coordinates from the set $\{0, 1, 2, 3\}$ by the rule — replace the coordinate value with the number of the quartile $(0, 1, 2, 3)$ it falls into.

**Quiz:** What coordinates will the vector $[2.3, 5.1, 2.0, 3.3]$ have if the quartile boundaries are: $([0, 2.1], [2.2, 3], [3.1, 4], [4.1, 5.0])$?

Submit the answer for coordinate number 2 (numbering starts from one).


In [ ]:
discretiser = fatf_discretisation.QuartileDiscretiser(
    X_train,
    feature_names=iris_feature_names)

data_point_discretised = discretiser.discretise(data_point)
sampled_data_discretised = discretiser.discretise(sampled_data)

discretiser.feature_bin_boundaries

Let's make sure that the values of the boundaries are really quartiles.

In [ ]:
import pandas as pd

pd.DataFrame(X_train, columns=iris_feature_names).describe()[3:] #Indeed, the boundaries coincide with the statistics we need.

In [ ]:
#Let's look at the discretized point
data_point_discretised

Now, to obtain an *interpretable model*, we will perform **binarization of the data**.
Binarization of data, as the name suggests, means converting all coordinates to a combination of zeros and ones (binary form). We will binarize using the following rule: set 0 if the coordinates of the discretized data point do not match the data point from the sample, and 1 otherwise.


In [ ]:
import fatf.utils.data.transformation as fatf_transformation

sampled_data_binarised = fatf_transformation.dataset_row_masking(
    sampled_data_discretised, data_point_discretised)

fatf_transformation.dataset_row_masking(data_point_discretised.reshape(1, -1), data_point_discretised)


In practice, binarization is applied to text data and images. In our example, it will produce results that may be inconsistent with other interpretation approaches. Which approach is better should be verified in practice.

```
**Remember:** Discretization and binarization also allow for more versatile use of a single explanation algorithm.
```


Let's look at all 3 transformations again:

In [ ]:
print(f'Original data point: {data_point}')
print(f'Discretized data point: {data_point_discretised}')

print(f'A sample from discretized data points before binarization: {sampled_data_discretised[7]}')
print(f'A sample from discretized data points after binarization: {sampled_data_binarised[7]}')

As the last step, it remains to calculate the weights of the objects for the error function. We will simply do this according to the appropriate formula that you recalled in the task above.

In [ ]:
import fatf.utils.distances as fatf_distances
import fatf.utils.kernels as fatf_kernels

features_number = sampled_data_binarised.shape[1]
kernel_width = np.sqrt(features_number) * 0.75

distances = fatf_distances.euclidean_point_distance(np.ones(features_number), sampled_data_binarised) #distances are calculated based on binarized data
weights = fatf_kernels.exponential_kernel(
     distances, width=kernel_width)

## **Training the Local Algorithm**

The task addressed by models for the Fisher's Iris dataset is a multi-class classification task. Therefore, when training the surrogate model, we can train it in two ways:
- Using the ONE VS REST approach, where the surrogate model will predict 0 or 1 depending on whether the object belongs to the class of interest.
- Using the classical approach, where the surrogate model will predict a probability vector.

The advantage of the first approach is the focus on the class of interest, while the second approach focuses on universality — the obtained surrogate can be used to explain objects from any class.

Additionally, we mentioned earlier that the model can be trained on binarized or non-binarized data. Training on binarized data results in a model that answers the question:

*"If this specific feature value of the explained data point were outside the range (for numerical features) or had a different value (for categorical features), how would it affect the probability of belonging to the explained class (probability classification) / predicted numerical value (regression)?"*

This is useful for images, as it allows comparison of the importance of different segments of the image.

In our case, the model will solve the task of minimizing the predicted values from the true ones in the classical sense.

```
**Remember:** The distance of points from the explained object also allows for tuning the explanation.
```


In [ ]:
sampled_data_predictions_versicolor = sampled_data_probabilities[:, 1] #let's save the probabilities for the OVR approach

Initialize the model and train it on discretized data on all probabilities.

In [ ]:
import sklearn.linear_model
import numpy as np

In [ ]:
lime_model = sklearn.linear_model.Ridge(alpha=1, fit_intercept=True)

lime_model.fit(sampled_data_discretised, sampled_data_predictions, sample_weight=weights)
for name, importance in zip(iris_feature_names, lime_model.coef_):
     print('->{}<-: {}'.format(name, importance))

** And now you have trained the first LIME!**
Let's fix the important signs in order: `petal width`, `petal length`

Let's look at the implemented version on non-discretized data

In [ ]:
lime_model = sklearn.linear_model.Ridge(alpha=1, fit_intercept=True)

lime_model.fit(sampled_data, sampled_data_predictions, sample_weight=weights)  #let's look at the implemented version on non-discretized data

for name, importance in zip(iris_feature_names, lime_model.coef_):
     print('->{}<-: {}'.format(name, importance))

Let's fix the important signs in order here: petal width, sepal lenght

**Quize: Change the approach of calculating distances — calculate them based on the original data values. Train a local model on discretized data. Have the 2 most important signs changed?**

In [ ]:
distances2 = fatf_distances.euclidean_point_distance(data_point, sampled_data) #calculate the distances based on the original data
weights2 = fatf_kernels.exponential_kernel(
     distances2, width=kernel_width)

lime_model = sklearn.linear_model.Ridge(alpha=1, fit_intercept=True)

lime_model.fit(sampled_data_discretised, sampled_data_predictions, sample_weight=weights2)   #let's look at the implemented version on discretized data

for name, importance in zip(iris_feature_names, lime_model.coef_):
     print('->{}<-: {}'.format(name, importance))

Answer:

**Some facts**
- Also, the top 2 signs will not change if the distances are calculated based on sampled data.
- Also, the top 2 attributes will not change if you build the model on the original data.


**Library Implementation**. \
Let's see what the library implementation gives us. All hyperparameters are quite intuitive, but let's explain each one just in case:

- `class_names` — names of the predicted classes
- `feature_names` — names of the features
- `kernel_width`— the width of the kernel, by default (and in our case) $\sqrt{n\_features}*0.75$
- `verbose` — detailed calculations during the training of the surrogate model
- `discretizer` — approach to discretization
- `mode` — the task solved by the model
- `discretize_continuous` — whether to discretize features
- `sample_around_instance` — whether to generate synthetic data **only in the vicinity** of the considered point

Let's sequentially look at the important features in discretized and non-discretized data. We'll start with non-discretized data.

```
**Remember:** The width of the neighborhood kernel also allows for tuning the explanation, but the default width is most commonly used.
```



In [ ]:
from lime.lime_tabular import LimeTabularExplainer #let's see what the library implementation provides

explainer = LimeTabularExplainer(X_train,
                                 class_names=iris_target_names,
                                 feature_names=iris_feature_names,
                                 kernel_width=np.sqrt(features_number) * 0.75,
                                 verbose=False,
                                 mode='classification',
                                 discretize_continuous=False,
                                 sample_around_instance=False)

exp = explainer.explain_instance(data_point, blackbox_model.predict_proba, top_labels=1)
exp.show_in_notebook()

We see that without discretization, there is no obvious force of influence of specific features.

**Quiz: Get an interpretation with quarterly sampling. Is it consistent with the one obtained above during manual implementation?**

In [ ]:
from lime.lime_tabular import LimeTabularExplainer #let's see what the library implementation provides

explainer = LimeTabularExplainer(X_train,
                                 class_names=iris_target_names,
                                 feature_names=iris_feature_names,
                                 kernel_width=np.sqrt(features_number) * 0.75,
                                 verbose=False,
                                 discretizer='quartile',
                                 mode='classification',
                                 discretize_continuous=True,
                                 sample_around_instance=False)

exp = explainer.explain_instance(data_point, blackbox_model.predict_proba, top_labels=1)
exp.show_in_notebook()

We see that the results are not exactly equal to each other, but they are similar in general conclusions.

**Quize: Get an interpretation using a manual implementation on binarized data on `weights' scales. What features (feature) are highlighted in this case?**

In [ ]:
distances2 = fatf_distances.euclidean_point_distance(data_point, sampled_data) #calculate the distances based on the original data
weights2 = fatf_kernels.exponential_kernel(
     distances2, width=kernel_width)

lime_model = sklearn.linear_model.Ridge(alpha=1, fit_intercept=True)

lime_model.fit(sampled_data_binarised, sampled_data_predictions, sample_weight=weights)  # let us look at the implemented variant on non-discretised data

for name, importance in zip(iris_feature_names, lime_model.coef_):
     print('->{}<-: {}'.format(name, importance))

**Answer: `petal width (cm)` and `petal length (cm)`**

## **The surrogate tree**

The linear model is not the only one in our arsenal. In some cases, building a surrogate tree is more useful and informative. A simple example of building:

In [ ]:
import sklearn.tree

blimey_tree = sklearn.tree.DecisionTreeClassifier(max_depth=3, random_state=42)
blimey_tree.fit(sampled_data, sampled_data_predictions, sample_weight=weights)


In [ ]:
for n_i in zip(iris_feature_names, blimey_tree.feature_importances_):
     name, importance = n_i
     print('->{}<-: {}'.format(name, importance))


It is also useful to visualize the partitioning rules for interpretation here.

In [ ]:
from sklearn import tree
print(tree.export_text(blimey_tree))

Or more beautiful.

In [ ]:
import graphviz

dot_data = tree.export_graphviz(blimey_tree, out_file=None,
                                feature_names=iris_feature_names,
                                class_names=iris_target_names,
                                filled=True)

# Draw graph
graph = graphviz.Source(dot_data, format="png")
graph

****Quiz: What feature is missing in the structure of the constructed surrogate tree?** \
Answer:

### **Conclusions**
- LIME allows for different approaches to feature importance formation and evaluation.
- Explanations within different LIME scenarios can vary, so the validity needs to be empirically verified, and hypotheses generated based on a combination of methods/models.
- Using LIME assumes that initially conceptually understandable features are used (image patches, numerical vector values, related to real data).
- Thanks to the flexibility of the algorithm's hyperparameters, you can analyze not only one explanation but also its stability depending on the data.

### **Hints to tuning LIME:**
1. The first thing you can change in LIME to get a different explanation is the vicinity of the point or the kernel width.
2. The second approach to tuning is methods for calculating the significance of the distance of objects to the point.
3. The width of the neighborhood kernel also allows for tuning the explanation, but the default width is most commonly used.
4. Discretization and binarization also allow for more versatile use of a single explanation algorithm, obtaining answers to other formulations of questions to the object.

Thank you for your time on this practice! I hope it will help to make your models more interpretable and reliable.

See you!

Your, \
Data Author

telegram: https://t.me/sabrina_sadiekh \
[LinkedIn](https://www.linkedin.com/in/sabrina-sadiekh-35181a286/) \
My course about explainable AI: https://open-xai-platform.web.app \